In [ ]:
import os
import asyncio
import base64
import mimetypes
from typing import List, Optional

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceInferenceAPIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.document_compressors import FlashrankRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableConfig

from langsmith import traceable
from docx import Document as DocxDocument

load_dotenv()

# =========================
# OUTPUT SCHEMA
# =========================
class RAGResponse(BaseModel):
    answer: str
    citations: List[str]
    confidence: float
    memory_update: Optional[str]


# =========================
# SECURITY
# =========================
def sanitize(text):
    blocked = ["ignore previous instructions", "system prompt", "override"]
    for b in blocked:
        text = text.replace(b, "")
    return text


# =========================
# ENGINE
# =========================
class EnterpriseMultimodalRAG:

    def __init__(self):
        self.llm = ChatGroq(
            model_name="meta-llama/llama-4-scout-17b-16e-instruct",
            temperature=0
        )

        self.embeddings = HuggingFaceInferenceAPIEmbeddings(
            model_name="BAAI/bge-small-en-v1.5"
        )

        self.parser = PydanticOutputParser(pydantic_object=RAGResponse)

        self.memory_path = "memory"
        self.mid_memory = "User profile empty"
        self._init_memory()

    # =========================
    # MEMORY SYSTEM
    # =========================
    def _init_memory(self):
        if os.path.exists(self.memory_path):
            self.memory = FAISS.load_local(
                self.memory_path,
                self.embeddings,
                allow_dangerous_deserialization=True
            )
        else:
            self.memory = FAISS.from_texts(["User initialized"], self.embeddings)

    def is_valid_memory(self, text):
        banned = ["patient", "diagnosis", "disease", "legal case", "treatment"]
        return not any(b in text.lower() for b in banned)

    @traceable(name="Memory Extraction")
    async def extract_structured_memory(self, history):
        prompt = f"""
Extract ONLY factual user preferences.
No assumptions. No roles.

Conversation:
{history[-5:]}
"""
        try:
            res = await self.llm.ainvoke(prompt)
            return res.content
        except:
            return None

    def update_memory(self, text):
        if not text or len(text) > 200:
            return

        existing = self.memory.similarity_search(text, k=1)
        if existing and text in existing[0].page_content:
            return

        self.memory.add_texts([text])
        self.memory.save_local(self.memory_path)

    def get_long_memory(self, query):
        docs = self.memory.similarity_search(query, k=3)
        return "\n".join([d.page_content for d in docs])

    async def update_mid_memory(self, history):
        prompt = f"Summarize user profile:\n{history[-5:]}"
        res = await self.llm.ainvoke(prompt)
        self.mid_memory = f"{self.mid_memory}\n{res.content}"

    # =========================
    # IMAGE PIPELINE
    # =========================
    def encode_image(self, path):
        mime, _ = mimetypes.guess_type(path)
        mime = mime or "image/jpeg"
        with open(path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        return {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}}

    @traceable(name="Image Processing")
    async def extract_image_info(self, image_path):
        try:
            res = await self.llm.ainvoke([
                HumanMessage(content=[
                    {"type": "text", "text": "Extract all details, labels, numbers, and insights from this image."},
                    self.encode_image(image_path)
                ])
            ])
            return res.content
        except:
            return None

    def store_image_info(self, caption, source):
        if not caption:
            return

        doc = Document(
            page_content=f"Image info: {caption}",
            metadata={"source": source, "page": 0}
        )

        self.memory.add_documents([doc])
        self.memory.save_local(self.memory_path)

    # =========================
    # MULTI-FORMAT INGESTION
    # =========================
    @traceable(name="Document Ingestion")
    def ingest(self, file_path):

        ext = file_path.split(".")[-1].lower()
        docs = []

        if ext == "pdf":
            from langchain_community.document_loaders import PyPDFLoader
            docs = PyPDFLoader(file_path).load()

        elif ext in ["txt", "md"]:
            with open(file_path, "r", encoding="utf-8") as f:
                docs = [Document(page_content=f.read(), metadata={"source": file_path})]

        elif ext in ["csv", "xlsx"]:
            df = pd.read_csv(file_path) if ext == "csv" else pd.read_excel(file_path)
            for i, row in df.iterrows():
                content = ", ".join([f"{col}: {row[col]}" for col in df.columns])
                docs.append(Document(page_content=content, metadata={"source": file_path, "row": i}))

        elif ext == "docx":
            doc = DocxDocument(file_path)
            text = "\n".join([p.text for p in doc.paragraphs])
            docs = [Document(page_content=text, metadata={"source": file_path})]

        else:
            raise ValueError("Unsupported format")

        if ext in ["pdf", "txt", "md", "docx"]:
            splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
            docs = splitter.split_documents(docs)

        vs = FAISS.from_documents(docs, self.embeddings)
        bm25 = BM25Retriever.from_documents(docs)

        ensemble = EnsembleRetriever(
            retrievers=[bm25, vs.as_retriever()],
            weights=[0.4, 0.6]
        )

        reranker = FlashrankRerank(top_n=3)

        return ContextualCompressionRetriever(
            base_retriever=ensemble,
            base_compressor=reranker
        )

    # =========================
    # GENERATION
    # =========================
    async def safe_generate(self, messages, config=None):
        for _ in range(2):
            try:
                res = await self.llm.ainvoke(messages, config=config)
                return self.parser.parse(res.content)
            except:
                messages[-1].content[0]["text"] += "\nFix JSON output."
        return RAGResponse(answer="Failed", citations=[], confidence=0, memory_update=None)

    # =========================
    # MAIN PIPELINE
    # =========================
    @traceable(name="RAG Pipeline")
    async def run(self, query, retriever, history, image=None):

        config = RunnableConfig(
            tags=["rag", "multimodal"],
            metadata={"query": query, "has_image": bool(image)}
        )

        query = sanitize(query)

        if len(history) % 5 == 0:
            await self.update_mid_memory(history)

        short_mem = "\n".join([f"{m['role']}: {m['content']}" for m in history[-5:]])
        long_mem = self.get_long_memory(query)

        docs = await retriever.ainvoke(query, config=config)

        if not docs:
            return RAGResponse(answer="No data found", citations=[], confidence=0, memory_update=None)

        context = "\n\n".join([
            f"{d.metadata.get('source')}|{d.metadata.get('page', d.metadata.get('row', 0))}:\n{sanitize(d.page_content)}"
            for d in docs
        ])

        valid_citations = [
            f"{d.metadata.get('source')}|{d.metadata.get('page', d.metadata.get('row', 0))}"
            for d in docs
        ]

        content = [{
            "type": "text",
            "text": f"""
Mid Memory:
{self.mid_memory}

Short Memory:
{short_mem}

Long Memory:
{long_mem}

Context:
{context}

Question:
{query}

{self.parser.get_format_instructions()}
"""
        }]

        if image:
            image_info = await self.extract_image_info(image)

            if image_info:
                self.store_image_info(image_info, image)
                content[0]["text"] += f"\n\nImage Info:\n{image_info}"

            content.append(self.encode_image(image))

        messages = [
            SystemMessage(content="Use ONLY provided context."),
            HumanMessage(content=content)
        ]

        result = await self.safe_generate(messages, config=config)

        result.citations = [c for c in result.citations if c in valid_citations]

        result.confidence = round(
            len(result.citations) / max(1, len(valid_citations)), 2
        )

        structured_memory = await self.extract_structured_memory(history)
        if structured_memory and self.is_valid_memory(structured_memory):
            self.update_memory(structured_memory)

        return result